# FactIndic 03 – Fine-tuned summarisers (mT5, IndicBART) and LoRA

Summarises every test input (mono, cm, roman, natural) with `csebuetnlp/mT5_multilingual_XLSum` and `ai4bharat/IndicBART-XLSum`, then trains a LoRA adapter on the mT5 checkpoint with the training articles in all three variants and evaluates it the same way (`mt5-lora-cm`). If IndicBART fails to load with your library versions the notebook continues without it.

| Setting (right-hand panel) | Value |
|---|---|
| Accelerator | **GPU T4 (one is enough)** |
| Internet | **On** |
| Inputs to attach | output of notebook 02 |
| Time (full profile) | 30–45 min |

Run it with **Save Version → Save & Run All (Commit)**. When it finishes, its output (folder `factindic_out`) is
ready to attach to the next notebook. Each notebook's output contains every earlier stage, so you only ever attach
the previous notebook.

In [1]:
# ── Setup: make the factindic code importable ─────────────────────────────────────────
# Kaggle: upload factindic-code.zip as a private dataset and attach it
#         (right-hand panel -> Add Input -> Your Work / Datasets -> factindic-code -> "+").
# Colab:  put factindic-code.zip in My Drive (and run the Google Drive cell first), or upload it to /content.
# Or:     push the repository to GitHub and paste its URL here (needs Internet).
REPO_URL = ""   # e.g. "https://github.com/<your-username>/factindic"

import glob, os, shutil, subprocess, sys, zipfile

ON_COLAB = "google.colab" in sys.modules
SRC = "/content/factindic_src" if ON_COLAB else "/kaggle/working/factindic_src"

def _is_pkg(d):
    return all(os.path.exists(os.path.join(d, f)) for f in ("__init__.py", "pipeline.py", "config.py"))

def _candidates():
    inits = glob.glob("/kaggle/input/**/__init__.py", recursive=True)
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    if ON_COLAB:  # shallow search only: Google Drive can be huge
        for base in ("/content", "/content/drive/MyDrive"):
            for depth in ("", "*/", "*/*/"):
                inits += glob.glob(f"{base}/{depth}factindic/__init__.py")
            zips += [z for z in glob.glob(f"{base}/*.zip") + glob.glob(f"{base}/*/*.zip")
                     if "factindic" in os.path.basename(z).lower()]
    return sorted(set(inits)), sorted(set(zips))

def _unzip(z):
    try:
        with zipfile.ZipFile(z) as zf:
            hits = [n for n in zf.namelist() if n.endswith("factindic/__init__.py")]
            if hits:
                shutil.rmtree(SRC, ignore_errors=True)
                zf.extractall(SRC)
                return os.path.dirname(os.path.dirname(os.path.join(SRC, sorted(hits, key=len)[-1])))
    except zipfile.BadZipFile:
        pass
    return None

def _find_code():
    inits, zips = _candidates()
    # 0) Colab: always unpack the newest zip from Drive, so replacing the zip there updates the code
    if ON_COLAB:
        for z in sorted(zips, key=os.path.getmtime, reverse=True):
            code = _unzip(z)
            if code:
                print("using", z)
                return code
    # 1) the repository folder (…/factindic/factindic/__init__.py)
    for p in inits:
        d = os.path.dirname(p)
        if _is_pkg(d) and os.path.basename(d) == "factindic":
            return os.path.dirname(d)
    # 2) only the inner package was uploaded (files directly in the dataset): copy it under the right name
    for p in inits:
        d = os.path.dirname(p)
        if _is_pkg(d):
            shutil.copytree(d, os.path.join(SRC, "factindic"), dirs_exist_ok=True)
            for req in glob.glob(os.path.join(os.path.dirname(d), "**", "requirements.txt"), recursive=True)[:1]:
                shutil.copy(req, SRC)
            return SRC
    # 3) a zip that nobody unpacked
    for z in zips:
        code = _unzip(z)
        if code:
            return code
    # 4) running locally from the repository
    for p in ("factindic", "../factindic"):
        if _is_pkg(p):
            return os.path.abspath(os.path.dirname(p) or ".")
    return None

CODE = _find_code()
if CODE is None and REPO_URL:
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, SRC], check=True)
    CODE = SRC
if CODE is None:
    where = (sorted(glob.glob("/content/*")) + sorted(glob.glob("/content/drive/MyDrive/*"))[:30]) if ON_COLAB else \
            (sorted(glob.glob("/kaggle/input/*")) + sorted(glob.glob("/kaggle/input/*/*"))[:30])
    how = ("Put factindic-code.zip in My Drive and run the Google Drive cell first (or upload the zip to /content)."
           if ON_COLAB else
           "Attach the dataset: right-hand panel -> Add Input -> Your Work (or Datasets) -> factindic-code -> '+'. "
           "If you uploaded it a minute ago, wait until Kaggle finishes processing it.")
    raise FileNotFoundError("factindic code not found.\n" + how + "\nWhat this notebook can see:\n  "
                            + ("\n  ".join(where) if where else "(nothing)"))
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
req = os.path.join(CODE, "requirements.txt")
pkgs = ["-r", req] if os.path.exists(req) else ["indic-transliteration", "wordfreq", "datasketch", "bert-score",
                                                "python-docx", "feedparser", "trafilatura", "peft", "openpyxl",
                                                "bitsandbytes"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
import factindic
from factindic import Config
from factindic import pipeline as P
print("factindic", factindic.__version__, "loaded from", CODE)

factindic 1.0.0 loaded from /kaggle/input/datasets/lkr1196/factindic-code/factindic


In [2]:
# ── Configuration ────────────────────────────────────────────────────────────────────
# Use the SAME profile in every notebook of the chain.
#   "full"  – the real experiment
#   "quick" – ~25 articles per language, to check the whole chain on Kaggle first
#   "mock"  – CPU dry run with stand-in models (no GPU, no downloads); never report its numbers
PROFILE = "full"
cfg = Config.make(PROFILE)
# cfg.seq2seq_models = ("mt5",)   # drop IndicBART
# cfg.lora = False
print(cfg.to_json())

{
  "profile": "full",
  "seed": 13,
  "langs": [
    "hi",
    "bn"
  ],
  "variants": [
    "mono",
    "cm",
    "roman"
  ],
  "data_source": "xlsum",
  "n_test": 200,
  "n_dev": 30,
  "n_train_cm": 60,
  "min_words": 120,
  "max_words_raw": 1200,
  "lead_words": 300,
  "max_sents": 14,
  "min_summary_words": 8,
  "max_summary_words": 60,
  "near_dup_threshold": 0.8,
  "natural_split": [
    0.7,
    0.1,
    0.2
  ],
  "scrape": true,
  "n_natural_per_lang": 100,
  "backend": "auto",
  "summarizer_model": "Qwen/Qwen3-4B-Instruct-2507",
  "codemix_model": "Qwen/Qwen3-4B-Instruct-2507",
  "openai_base_url": "",
  "openai_model": "",
  "openai_concurrency": 4,
  "max_model_len": 6144,
  "max_batch_tokens": 24000,
  "max_batch_size": 24,
  "repetition_penalty": 1.0,
  "workers": -1,
  "quantize": "auto",
  "summary_max_words": 40,
  "tracks": [
    "A",
    "B",
    "BV"
  ],
  "max_new_tokens_A": 448,
  "max_new_tokens_B": 1536,
  "seq2seq_models": [
    "mt5",
    "indicbart"
  ],
 

In [3]:
s2s = P.run_stage03(cfg)

Stages available: 01_data, 02_codemix
== mt5: csebuetnlp/mT5_multilingual_XLSum


config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spiece.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

... (21 outputs left out)

LoRA: 3.54M trainable parameters, 270 training pairs
  epoch 1: train loss 3.266, dev loss 2.669
  epoch 2: train loss 2.463, dev loss 2.332


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


  mt5-lora-cm bn: 661 summaries in 5.6 min
  mt5-lora-cm hi: 672 summaries in 5.8 min


In [4]:
s2s.groupby(["system", "lang", "variant"]).agg(n=("summary", "count"), time_s=("time_s", "mean"))

n    time_s
system      lang variant               
indicbart   bn   cm       200  0.385030
                 mono     200  0.386215
                 natural   61  0.356189
                 roman    200  0.402618
            hi   cm       200  0.247515
                 mono     200  0.258454
                 natural   72  0.221783
                 roman    200  0.339052
mt5         bn   cm       200  0.448281
                 mono     200  0.441612
                 natural   61  0.427298
                 roman    200  0.478046
            hi   cm       200  0.455166
                 mono     200  0.458565
                 natural   72  0.451213
                 roman    200  0.475069
mt5-lora-cm bn   cm       200  0.516918
                 mono     200  0.496836
                 natural   61  0.492781
                 roman    200  0.500231
            hi   cm       200  0.504662
                 mono     200  0.513600
                 natural   72  0.504935
                 roman    200  0.527187

In [5]:
s2s.groupby("system").head(2)[["system", "lang", "variant", "summary"]]

,system,lang,variant,summary
0,mt5,bn,mono,বাংলাদেশের জনপ্রিয় কথা শিল্পী হুমায়ূন আহমেদক...
1,mt5,bn,mono,"যুক্তরাষ্ট্রের এক গবেষণায় বলা হয়েছে যে, গত ক..."
1333,indicbart,bn,mono,"হুমায়ুন আহমেদ বাংলাদেশ সময় দুপুর দুইটায়, নু..."
1334,indicbart,bn,mono,পরিবারগুলো সম্পর্কে সঠিক তথ্য পাওয়া যায় না ।...
2666,mt5-lora-cm,bn,mono,বাংলাদেশের জনপ্রিয় কথা শিল্পী হুমায়ূন আহমেদক...
2667,mt5-lora-cm,bn,mono,"যুক্তরাষ্ট্রের এক গবেষণা অনুযায়ী, দেশে বিবাহ-..."


In [6]:
import os
for root, dirs, files in os.walk(cfg.out_root):
    dirs[:] = [d for d in dirs if not d.startswith("_")]
    depth = root[len(cfg.out_root):].count(os.sep)
    if depth <= 1:
        print("  " * depth + os.path.basename(root) + "/", ", ".join(sorted(files))[:300])

factindic_out/ 
  03_seq2seq/ config.json, lora_training.json, summaries_seq2seq.jsonl
  01_data/ config.json, data_report.md, data_stats.csv, docs.jsonl
  02_codemix/ codemix_quality_check.xlsx, codemix_report.md, config.json, env.json, variants.jsonl
